# Step 7: Spark Data Cleaning (Deep) - Synapse Edition
Advanced cleaning including fuzzy column mapping and Z-score quality scoring. **Environment: Azure Synapse Analytics**.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from fuzzywuzzy import process
import json
import re
import unicodedata

# 1. Synapse Storage Configuration
storage_account = "fileuploadsa35b5"
container_name = "normalized-json"
base_path = f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/"
output_path = f"abfss://processed@{storage_account}.dfs.core.windows.net/cleaned_data"

# 2. Load Data
df = spark.read.format("delta").load(base_path)

# 3. Fuzzy Column Reconciliation
canonical_map = {
    "customer_id": ["cust_id", "client_id", "user_id"],
    "amount": ["total_value", "price", "cost", "sum"],
    "created_at": ["date", "timestamp", "created"],
    "description": ["memo", "note", "text", "comment"]
}
current_cols = df.columns
for canonical, variants in canonical_map.items():
    best_match, score = process.extractOne(canonical, current_cols)
    if score > 85 and best_match not in canonical_map.keys():
        df = df.withColumnRenamed(best_match, canonical)
    else:
        for v in variants:
            if v in current_cols: df = df.withColumnRenamed(v, canonical); break

# 4. Cleaning Logic (Dates & Text)
date_cols = [f.name for f in df.schema.fields if "date" in f.name.lower() or "timestamp" in f.name.lower()]
for c in date_cols: df = df.withColumn(c, F.coalesce(*[F.to_date(c, f) for f in ["dd/MM/yyyy", "yyyy-MM-dd", "yyyy.MM.dd"]]))

# 5. Statistical Quality Score
df = df.withColumn("row_quality_score", F.lit(100.0))
numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, (DoubleType, FloatType, LongType, IntegerType))]
for col in numeric_cols:
    stats = df.select(F.mean(col).alias("mu"), F.stddev(col).alias("sigma")).collect()[0]
    if stats['sigma'] and stats['sigma'] > 0:
        df = df.withColumn("row_quality_score", F.col("row_quality_score") - (F.when(F.abs((F.col(col)-stats['mu'])/stats['sigma']) > 3, 20).otherwise(0)))

# 6. DISPLAY INSIGHTS (Synapse Specific)
display(df.limit(10))

# 7. Save
df.write.format("delta").mode("overwrite").save(output_path)